# SpectraLite

## Phase 0 — Environment Setup & Verification

This notebook validates the SpectraLite research environment. It does **not** implement compression, SVD, whitening, rank allocation, or latency engineering.

**Objectives**
1. Install Phase 0 dependencies (**without overwriting Colab CUDA PyTorch**)
2. Verify Python / PyTorch / CUDA / GPU visibility
3. Load `facebook/opt-125m` (FP16 on CUDA, `device_map="auto"`)
4. Inventory every `nn.Linear` layer for later SVD targeting
5. Run one short generation smoke test
6. Report GPU memory before load / after load / after inference

All logic lives in the `spectralite` package; cells below only orchestrate the experiment.

> **Colab:** Runtime → Change runtime type → **GPU (A100)** → Save, *then* run. After pulling code updates: `Runtime → Restart session`, then run from the top.


### Colab Bootstrap (clone / refresh repo)

Run this cell **first** on Colab so `spectralite/` is on disk. Skip on local if the repo is already checked out.


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/PrabinDevkota/Execution.git"
CLONE_ROOT = Path("/content/Execution")
PACKAGE_ROOT = CLONE_ROOT / "SpectraLite"

if IN_COLAB:
    if not PACKAGE_ROOT.is_dir():
        !git clone {REPO_URL} {CLONE_ROOT}
    else:
        # Pull latest fixes (e.g. Colab-safe dependency install)
        %cd {CLONE_ROOT}
        !git pull --ff-only
    %cd {PACKAGE_ROOT}
    if str(PACKAGE_ROOT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_ROOT))
    print("Colab repo ready:", PACKAGE_ROOT)
    print("spectralite exists:", (PACKAGE_ROOT / "spectralite").is_dir())
else:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent]:
        if (candidate / "spectralite").is_dir():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            print("Local repo ready:", candidate)
            break


Cloning into '/content/Execution'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 50 (delta 15), reused 32 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 42.14 KiB | 21.07 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Execution/SpectraLite
Colab repo ready: /content/Execution/SpectraLite
spectralite exists: True


### Install Dependencies

On **Colab**, installs `requirements-colab.txt` (no `torch`) so the runtime CUDA build is preserved.
On **local**, installs full `requirements.txt` (includes `torch`).


In [2]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, Path("/content/Execution/SpectraLite")]
repo_root = None
for candidate in candidates:
    if (candidate / "spectralite").is_dir() and (candidate / "requirements.txt").is_file():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate SpectraLite repo root (expected spectralite/ + requirements.txt)."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

req_name = "requirements-colab.txt" if IN_COLAB else "requirements.txt"
requirements = repo_root / req_name
if not requirements.is_file():
    raise FileNotFoundError(f"Missing {requirements}")

print(f"Repo root      : {repo_root}")
print(f"Colab detected : {IN_COLAB}")
print(f"Requirements   : {requirements}")

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
)

# Sanity check: Colab must keep a CUDA-capable torch
try:
    import torch
    print(f"Torch version  : {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")
    if IN_COLAB and not torch.cuda.is_available():
        print(
            "WARNING: Colab has no CUDA visible. "
            "Runtime → Change runtime type → GPU (A100), then Restart session and re-run."
        )
    elif IN_COLAB and "+cpu" in torch.__version__.lower():
        print(
            "WARNING: CPU-only torch wheel detected on Colab. "
            "Runtime → Disconnect and delete runtime, select GPU, then re-run without installing torch."
        )
except ImportError:
    print("WARNING: torch not importable yet")

print("Dependencies installed.")


Repo root      : /content/Execution/SpectraLite
Colab detected : True
Requirements   : /content/Execution/SpectraLite/requirements-colab.txt
Torch version  : 2.11.0+cu128
CUDA available : True
Dependencies installed.


### Environment Verification

Adds the repo to `sys.path`, seeds RNGs, ensures output directories exist, and prints the Phase 0 environment report (Python, PyTorch, CUDA, GPU).


In [3]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, Path("/content/Execution/SpectraLite")]:
    if (candidate / "spectralite").is_dir():
        repo_root = candidate
        break
assert repo_root is not None, "SpectraLite repo root not found"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from spectralite import __version__, default_config, set_seed
from spectralite.system import print_environment_report
from spectralite.utils import get_logger, print_section

cfg = default_config()
cfg.ensure_directories()
set_seed(cfg.seed)
logger = get_logger("spectralite.phase0", level=cfg.log_level)

print_section("SpectraLite Phase 0")
print(f"  Package version : {__version__}")
print(f"  Model           : {cfg.model_name}")
print(f"  Seed            : {cfg.seed}")
print(f"  Dtype preference: {cfg.dtype}")
print(f"  Device map      : {cfg.device_map}")

env_info = print_environment_report()
environment_ready = env_info["pytorch_version"] != "unavailable"
gpu_ready = bool(env_info["cuda_available"])
logger.info("Environment ready=%s | GPU ready=%s", environment_ready, gpu_ready)

if not gpu_ready:
    print("\n  >>> GPU NOT READY. For SpectraLite latency work you need a CUDA runtime.")
    print("  >>> Colab: Runtime → Change runtime type → GPU (A100) → Restart → re-run from top.")



 SpectraLite Phase 0
  Package version : 0.1.0-phase0
  Model           : facebook/opt-125m
  Seed            : 42
  Dtype preference: float16
  Device map      : auto

 Environment Verification
  Python Version               3.12.13
  PyTorch Version              2.11.0+cu128
  CUDA Version                 12.8
  cuDNN Version                91900
  GPU Name                     NVIDIA A100-SXM4-40GB
  GPU Memory                   39.49 GB
  Torch CUDA Available         True
  Device                       cuda
  GPU Count                    1
  Platform                     Linux-6.6.122+-x86_64-with-glibc2.35
2026-07-12 15:13:19 | INFO     | spectralite.phase0 | Environment ready=True | GPU ready=True


### GPU Information

Captures a baseline GPU memory snapshot **before** the model is loaded. On CPU-only machines this section reports that CUDA is unavailable and continues.


In [4]:
from spectralite import gpu

mem_before = gpu.snapshot(label="Memory before loading")
gpu.print_memory_snapshot(mem_before)

print("\n  CUDA available :", gpu.is_cuda_available())
if gpu.is_cuda_available():
    print("  Total VRAM     :", mem_before["total"])
    print("  Free (approx)  :", mem_before["free"])



 Memory before loading
  Allocated                    0 B
  Reserved                     0 B
  Free (approx)                39.49 GB
  Total                        39.49 GB

  CUDA available : True
  Total VRAM     : 39.49 GB
  Free (approx)  : 39.49 GB


### Load Model

Loads `facebook/opt-125m` via `AutoTokenizer` + `AutoModelForCausalLM`.

- **Dtype:** FP16 when CUDA is available, otherwise FP32
- **Placement:** `device_map="auto"`
- **Mode:** `eval()` (no training in Phase 0)


In [5]:
from spectralite.model_loader import (
    load_model_and_tokenizer,
    print_model_load_summary,
)

model, tokenizer = load_model_and_tokenizer(config=cfg)
load_summary = print_model_load_summary(model, model_name=cfg.model_name)

mem_after_load = gpu.snapshot(label="Memory after loading")
gpu.print_memory_snapshot(mem_after_load)

model_loaded = model is not None and tokenizer is not None
logger.info("Model loaded: %s on %s", cfg.model_name, load_summary["device"])


2026-07-12 15:13:38 | INFO     | spectralite.model_loader | Loading tokenizer: facebook/opt-125m


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

2026-07-12 15:13:41 | INFO     | spectralite.model_loader | Loading model: facebook/opt-125m | dtype=torch.float16 | device_map=auto


pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


 Model Load Summary
  Model Name                   facebook/opt-125m
  Architecture                 OPTForCausalLM
  Total Parameters             125.24M (125,239,296)
  Trainable Parameters         125.24M (125,239,296)
  Model Size (params)          238.88 MB
  Tensor Data Type             float16
  Current Device               cuda:0

 Memory after loading
  Allocated                    241.50 MB
  Reserved                     358.00 MB
  Free (approx)                39.14 GB
  Total                        39.49 GB
2026-07-12 15:13:48 | INFO     | spectralite.phase0 | Model loaded: facebook/opt-125m on cuda:0


### Model Analysis

Prints architecture statistics and **every** `nn.Linear` layer (name, in/out features, weight shape). This inventory is the targeting surface for later SVD compression phases — Phase 0 only inspects.


In [6]:
from spectralite.model_analysis import print_full_model_analysis

analysis = print_full_model_analysis(model, model_name=cfg.model_name)

print(f"\n  Block names (first 3): {analysis['block_names'][:3]}")
print(f"  Linear layers catalogued: {analysis['num_linear_layers']}")



 Model Information
  Model Name                   facebook/opt-125m
  Architecture                 OPTForCausalLM
  Transformer Blocks           12
  Attention Linear Layers      48
  MLP Linear Layers            24
  Other Linear Layers          1
  Total nn.Linear Layers       73
  Total Parameters             125.24M (125,239,296)
  Trainable Parameters         125.24M (125,239,296)
  Model Size / Memory Footprint 238.88 MB
  Tensor Data Type             float16
  Current Device               cuda:0

 nn.Linear Inventory (73 layers)
  #    Layer Name                                                  In    Out Weight Shape    
  -------------------------------------------------------------------------------------------
  0    model.decoder.layers.0.self_attn.k_proj                    768    768 (768, 768)      
  1    model.decoder.layers.0.self_attn.v_proj                    768    768 (768, 768)      
  2    model.decoder.layers.0.self_attn.q_proj                    768    768 (768

### Test Inference

Single smoke-test generation to confirm the forward / generate path works end-to-end.

- **Prompt:** *Explain Singular Value Decomposition in one sentence.*
- **Budget:** ~50 new tokens (greedy decoding for reproducibility)


In [7]:
from spectralite.model_loader import generate_text
from spectralite.utils import print_section

prompt = cfg.smoke_prompt
generated = generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=cfg.max_new_tokens,
    do_sample=False,
)

print_section("Test Inference")
print("  Prompt:")
print(f"    {prompt}")
print("\n  Model output:")
print("  " + "-" * 60)
for line in generated.strip().splitlines() or [generated.strip()]:
    print(f"  {line}")
print("  " + "-" * 60)

inference_ok = isinstance(generated, str) and len(generated.strip()) > 0
logger.info("Inference successful=%s | chars=%d", inference_ok, len(generated))



 Test Inference
  Prompt:
    Explain Singular Value Decomposition in one sentence.

  Model output:
  ------------------------------------------------------------
  Explain Singular Value Decomposition in one sentence.
  
  I’m not sure what you mean by “explain”.
  
  I’m not sure what you mean by “explain”.
  
  I’m not sure what you mean by
  ------------------------------------------------------------
2026-07-12 15:13:58 | INFO     | spectralite.phase0 | Inference successful=True | chars=168


### GPU Memory

Final memory snapshot after inference, then a three-stage timeline (before load → after load → after inference).


In [8]:
mem_after_infer = gpu.snapshot(label="Memory after inference")
gpu.print_memory_snapshot(mem_after_infer)

gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])



 Memory after inference
  Allocated                    250.62 MB
  Reserved                     362.00 MB
  Free (approx)                39.14 GB
  Total                        39.49 GB

 GPU Memory Timeline
  Memory before loading        alloc=0 B          reserved=0 B          free=39.49 GB
  Memory after loading         alloc=241.50 MB    reserved=358.00 MB    free=39.14 GB
  Memory after inference       alloc=250.62 MB    reserved=362.00 MB    free=39.14 GB


### Summary

Phase 0 completion checklist. Aim for **GPU Ready = PASS** before Phase 1.


In [9]:
from spectralite.utils import print_checklist, print_section

phase0_complete = all([environment_ready, model_loaded, inference_ok])

print_checklist(
    [
        ("Environment Ready", environment_ready),
        ("GPU Ready", gpu_ready),
        ("Model Loaded Successfully", model_loaded),
        ("Inference Successful", inference_ok),
        ("Phase 0 Complete", phase0_complete),
    ]
)

print_section("Phase 0 Outcome")
print("  Environment Ready" if environment_ready else "  Environment NOT Ready")
print("  GPU Ready" if gpu_ready else "  GPU NOT Ready (select Colab GPU runtime and re-run)")
print("  Model Loaded Successfully" if model_loaded else "  Model Load FAILED")
print("  Inference Successful" if inference_ok else "  Inference FAILED")
print("  Phase 0 Complete" if phase0_complete else "  Phase 0 INCOMPLETE")

if not gpu_ready:
    raise SystemExit(
        "Phase 0 code path works, but GPU is not ready. "
        "Fix the Colab runtime (GPU/A100), restart, and re-run before Phase 1."
    )



 Phase 0 Status
  [PASS] Environment Ready
  [PASS] GPU Ready
  [PASS] Model Loaded Successfully
  [PASS] Inference Successful
  [PASS] Phase 0 Complete

 Phase 0 Outcome
  Environment Ready
  GPU Ready
  Model Loaded Successfully
  Inference Successful
  Phase 0 Complete
